# Open-Fret HPO monitor

Live view of the running Optuna study. Everything is read from disk:
- Study DB: `hpo-dev.db` (copied to /tmp first, so reads never lock the running study)
- Per-trial curves: `models/hpo-dev-v1/trial_<n>/trial_<n>_history.json`
- Summary: `models/hpo-dev-v1/batch_saturation.csv`

**To refresh, re-run all cells** (Kernel → Restart & Run All, or run cells top to bottom).

In [ ]:
import json
import shutil
import sqlite3
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import optuna
import pandas as pd

%matplotlib inline

REPO = Path.cwd() if (Path.cwd() / "hpo-dev.db").exists() else Path(__file__).parent.parent if "__file__" in globals() else Path.cwd()
# If the notebook lives in notebooks/, repo root is one level up:
if (REPO / "notebooks").exists() is False and REPO.name == "notebooks":
    REPO = REPO.parent
if not (REPO / "hpo-dev.db").exists():
    cand = Path.cwd()
    while cand != cand.parent and not (cand / "hpo-dev.db").exists():
        cand = cand.parent
    REPO = cand
print("repo:", REPO)

STUDY_NAME = "dev-hpo-v3"
DB_SRC = REPO / "hpo-dev.db"
OUT_ROOT = REPO / "models" / "hpo-dev-v1"

# Copy the live DB so reads never block (or get blocked by) the writer.
DB_COPY = Path(tempfile.gettempdir()) / "hpo-dev-readcopy.db"
shutil.copy2(DB_SRC, DB_COPY)
study = optuna.load_study(study_name=STUDY_NAME, storage=f"sqlite:///{DB_COPY}")
print(f"study '{STUDY_NAME}': {len(study.trials)} trials")

In [ ]:
# --- Trials table ---
rows = []
for t in study.trials:
    a = t.user_attrs
    rows.append({
        "trial": t.number, "state": t.state.name,
        "val_loss": t.value, "batch": t.params.get("batch_size"),
        "accum": t.params.get("grad_accum"),
        "lr": t.params.get("lr"),
        "warmup_ex": t.params.get("warmup_examples"),
        "decay_at": t.params.get("decay_after_examples"),
        "decay_x": t.params.get("decay_factor"),
        "peak_mem_gb": a.get("peak_mem_gb"),
        "avg_gpu%": a.get("avg_gpu_util_pct"),
        "stable": a.get("stable"),
        "samples/s": round(a.get("train_samples_per_second", 0) or 0),
    })
df = pd.DataFrame(rows).sort_values("trial")
df

In [ ]:
# --- Optimization history: best val_loss so far ---
done = df.dropna(subset=["val_loss"]).sort_values("trial")
if len(done):
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot(done["trial"], done["val_loss"], "o-", label="trial value")
    ax.plot(done["trial"], done["val_loss"].cummin(), "r--", label="best so far")
    ax.set_xlabel("trial"); ax.set_ylabel("val_loss"); ax.legend()
    ax.set_title("HPO progress (30-min time-capped trials)")
    plt.show()
else:
    print("no finished trials yet")

In [ ]:
# --- Eval-loss curves of the K best finished trials ---
K = 5
best_ids = done.nsmallest(min(K, len(done)), "val_loss")["trial"].tolist() if len(done) else []
fig, ax = plt.subplots(figsize=(9, 4))
for tid in best_ids:
    p = OUT_ROOT / f"trial_{tid}" / f"trial_{tid}_history.json"
    if not p.exists():
        continue
    h = json.loads(p.read_text())
    xs = [pt["step"] for pt in h.get("eval_loss_curve", [])]
    ys = [pt["eval_loss"] for pt in h.get("eval_loss_curve", [])]
    if xs:
        ax.plot(xs, ys, "o-", ms=3, label=f"trial {tid}")
ax.set_xlabel("step"); ax.set_ylabel("eval_loss"); ax.legend()
ax.set_title(f"Eval curves — {len(best_ids)} best trials")
plt.show() if best_ids else print("nothing to plot yet")

In [ ]:
# --- GPU utilization curves (recent trials) ---
RECENT = 4
recent = sorted([t for t in study.trials if "gpu_util_curve" in t.user_attrs],
              key=lambda t: t.number)[-RECENT:]
fig, axes = plt.subplots(1, max(len(recent), 1), figsize=(4 * max(len(recent), 1), 3.5), sharey=True)
if not isinstance(axes, (list, )):
    import numpy as _np
    axes = _np.atleast_1d(axes)
for ax, t in zip(axes, recent):
    curve = t.user_attrs["gpu_util_curve"]
    xs = [p["t"] / 60 for p in curve]
    ax.plot(xs, [p["util_pct"] for p in curve], label="util %")
    ax.set_xlabel("min"); ax.set_title(f"trial {t.number} (batch {t.params.get('batch_size')})")
    ax.set_ylim(0, 105)
axes[0].set_ylabel("GPU util %")
plt.suptitle(f"avg util: " + ", ".join(
    f"t{t.number}={t.user_attrs.get('avg_gpu_util_pct')}%%" for t in recent))
plt.show() if recent else print("no GPU curves stored yet")

In [ ]:
# --- Batch saturation: memory vs quality ---
sat_path = OUT_ROOT / "batch_saturation.csv"
sat_done = pd.DataFrame()
if sat_path.exists():
    _sat = pd.read_csv(sat_path)
    sat_done = _sat[_sat["state"] == "COMPLETE"].sort_values("batch_size")
    sat_done
else:
    print("batch_saturation.csv is written when the study finishes; "
          "use the trials table above until then")


In [ ]:
if len(sat_done):
    fig, ax = plt.subplots(figsize=(8, 4))
    sc = ax.scatter(sat_done["batch_size"], sat_done["val_loss"],
                    s=(sat_done["peak_mem_gb"].fillna(0) * 12).clip(lower=20),
                    c=sat_done["avg_gpu_util_pct"].fillna(0),
                    cmap="viridis", alpha=0.8)
    for _, r in sat_done.iterrows():
        ax.annotate(f"t{int(r['trial'])}", (r["batch_size"], r["val_loss"]),
                    fontsize=8, xytext=(4, 4), textcoords="offset points")
    ax.set_xlabel("batch_size"); ax.set_ylabel("val_loss")
    fig.colorbar(sc, label="avg GPU util %")
    ax.set_title("Bigger marker = more peak VRAM")
    plt.show()
else:
    print("(saturation scatter appears once trials complete)")


In [ ]:
# --- Best trial detail ---
try:
    b = study.best_trial
    print("best trial:", b.number, "val_loss:", round(b.value, 4))
    print(json.dumps(b.params, indent=2))
    print("attrs:", {k: v for k, v in b.user_attrs.items() if not k.endswith("_curve")})
except ValueError:
    print("no finished trials yet")